# Colab setup

Run these cells in order with Python 3.12. The scaffold does not download data or weights.
Optionally add `GITHUB_TOKEN` in Colab Secrets and grant this notebook access.
Existing torch/torchvision versions are constrained during installation so pip cannot replace them.


In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path

from google.colab import userdata

repo_url = "https://github.com/MichalTalalaj/lordosis-mri.git"
workspace = Path.cwd()
repo_dir = workspace if (workspace / ".git").is_dir() else workspace / "lordosis-mri"
try:
    github_token = userdata.get("GITHUB_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    github_token = None

clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
try:
    if not repo_dir.exists():
        with tempfile.TemporaryDirectory() as helper_dir:
            if github_token:
                askpass = Path(helper_dir) / "git_askpass.py"
                askpass.write_text(
                    "#!/usr/bin/env python3\n"
                    "import os, sys\n"
                    'print("x-access-token" if "username" in sys.argv[1].lower() '
                    'else os.environ["LORDOSIS_GITHUB_TOKEN"])\n',
                    encoding="utf-8",
                )
                askpass.chmod(0o700)
                clone_env["GIT_ASKPASS"] = str(askpass)
                clone_env["LORDOSIS_GITHUB_TOKEN"] = github_token
            subprocess.run(
                ["git", "-c", "credential.helper=", "clone",
                 "--branch", "main", repo_url, str(repo_dir)],
                env=clone_env,
                check=True,
            )
    elif not (repo_dir / ".git").is_dir():
        raise RuntimeError("The checkout path exists but is not a git repository.")
finally:
    github_token = None
    clone_env.pop("LORDOSIS_GITHUB_TOKEN", None)

os.chdir(repo_dir)


In [ ]:
import sys
from importlib.metadata import version

# Verify that Colab supplies both packages before installing anything.
torch_version = version("torch")
torchvision_version = version("torchvision")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True
)
with tempfile.TemporaryDirectory() as constraint_dir:
    constraints = Path(constraint_dir) / "colab-torch.txt"
    constraints.write_text(
        f"torch=={torch_version}\ntorchvision=={torchvision_version}\n", encoding="utf-8"
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt",
         "--constraint", str(constraints)],
        check=True,
    )


In [ ]:
from google.colab import drive

drive_root = Path.cwd().parent / "drive"
drive.mount(str(drive_root))
# Edit this relative Drive location to point to your future dataset directory.
data_root = drive_root / "MyDrive" / "lordosis-mri" / "data"
os.environ["LORDOSIS_DATA_ROOT"] = str(data_root)


In [ ]:
import lordosis
from lordosis.common.device import pick_device

print(lordosis.__name__, "import OK")
print("Device:", pick_device())
